# 🧰 Instructor notebook – export ONE study area (e.g. Gicumbi) to small files for GitHub
**Run this once** (it uses Google Earth Engine, so you need your EE project). It clips every dataset to one district, saves small GeoTIFF / CSV / GeoJSON files and gives you a **zip**.

Then: unzip → copy the folder into your GitHub repo as `data/gicumbi/` → commit & push. Students use **`Rwanda_GIS_GitHub_Student.ipynb`**, which reads these files directly from GitHub – **no Earth Engine, no sign-in, light on phones**.

Expected zip size: roughly 10–25 MB for a district (far below GitHub's 100 MB-per-file limit).

In [ ]:
#@title 1️⃣ Sign in to Earth Engine (instructor only)
EE_PROJECT = "" #@param {type:"string"}
!pip -q install osmnx rasterio
import ee, io, os, json, math, zipfile, shutil, requests, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
warnings.filterwarnings('ignore')
try:
    ee.Initialize(project=EE_PROJECT or None)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT or None)
print('✅ Earth Engine ready')

In [ ]:
#@title 2️⃣ Choose the district and save boundaries
DISTRICT = "Gicumbi" #@param {type:"string"}
AREA_FOLDER = "gicumbi" #@param {type:"string"}
#@markdown UTM zone for the rasters: Gicumbi (≈30°E) is on the 35S/36S edge – 36S is fine for eastern/northern areas, use 35S for the west (Rusizi, Rubavu…).
CRS = "EPSG:32736" #@param ["EPSG:32736","EPSG:32735"]
YEAR_FROM = 2001 #@param {type:"integer"}
YEAR_TO = 2023 #@param {type:"integer"}

DEST = f'/content/{AREA_FOLDER}'
shutil.rmtree(DEST, ignore_errors=True)
os.makedirs(DEST)

def gb(level):
    meta = requests.get(f'https://www.geoboundaries.org/api/current/gbOpen/RWA/{level}/', timeout=60).json()
    return gpd.read_file(meta['gjDownloadURL']).to_crs(4326)

dist = gb('ADM2')
m = dist[dist['shapeName'].str.lower().str.contains(DISTRICT.lower(), regex=False)]
if m.empty:
    raise ValueError('No match. Available: ' + ', '.join(sorted(dist['shapeName'])))
aoi_union = m.geometry.unary_union
aoi_gdf = gpd.GeoDataFrame({'shapeName': [DISTRICT]}, geometry=[aoi_union], crs=4326)
try:
    sect = gb('ADM3')
    sects = sect[sect.geometry.representative_point().within(aoi_union)][['shapeName', 'geometry']].copy()
except Exception:
    sects = gpd.GeoDataFrame({'shapeName': []}, geometry=[], crs=4326)
rw = dist[['shapeName', 'geometry']].copy()
rw['geometry'] = rw.geometry.simplify(0.005)

aoi_gdf.to_file(f'{DEST}/boundary_district.geojson', driver='GeoJSON')
sects.to_file(f'{DEST}/boundary_sectors.geojson', driver='GeoJSON')
rw.to_file(f'{DEST}/rwanda_districts.geojson', driver='GeoJSON')
g_s = gpd.GeoSeries([aoi_union.simplify(0.001)], crs=4326)
AOI = ee.Geometry(json.loads(g_s.to_json())['features'][0]['geometry'])
print(f'✅ {DISTRICT}: {len(sects)} sectors saved. Next: export rasters.')

In [ ]:
#@title 3️⃣ Export rasters (GeoTIFF, clipped to the district)
hf = None
HANSEN = None
for vid in ['UMD/hansen/global_forest_change_2025_v1_13', 'UMD/hansen/global_forest_change_2024_v1_12', 'UMD/hansen/global_forest_change_2023_v1_11']:
    try:
        hf = ee.Image(vid); hf.bandNames().getInfo(); HANSEN = vid; break
    except Exception:
        hf = None

gh = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S').select('built_surface').filter(ee.Filter.lte('system:time_start', ee.Date('2020-12-31').millis()))
ntl = ee.ImageCollection('NOAA/VIIRS/DNB/ANNUAL_V21').select('average')
pop_c = ee.ImageCollection('WorldPop/GP/100m/pop').filter(ee.Filter.eq('country', 'RWA'))

LAYERS = {
    'dem.tif': (ee.Image('USGS/SRTMGL1_003').select('elevation').toInt16(), 30, -9999),
    'worldcover.tif': (ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').toUint8(), 30, 255),
    'water_occurrence.tif': (ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').toUint8(), 30, 255),
    'treecover2000.tif': (hf.select('treecover2000').toUint8(), 30, 255),
    'lossyear.tif': (hf.select('lossyear').toUint8(), 30, 255),
    'soil_texture.tif': (ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').select('b0').toUint8(), 250, 255),
    'soil_ph_x10.tif': (ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02').select('b0').toUint8(), 250, 255),
    'pop_2020.tif': (pop_c.filter(ee.Filter.eq('year', 2020)).select('population').mosaic().toFloat(), 100, -9999),
    'built_2020.tif': (gh.sort('system:time_start', False).first().toFloat(), 100, -9999),
    'ntl_latest.tif': (ntl.sort('system:index', False).first().max(0).toFloat(), 500, -9999),
}

from rasterio.features import geometry_mask
AOI_UTM = list(aoi_gdf.to_crs(CRS).geometry)
for fname, (img, scale, nd) in LAYERS.items():
    url = img.clip(AOI).unmask(nd).getDownloadURL({'region': AOI, 'scale': scale, 'crs': CRS, 'format': 'GEO_TIFF'})
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    data = r.content
    if data[:2] == b'PK':
        z = zipfile.ZipFile(io.BytesIO(data))
        data = z.read([n for n in z.namelist() if n.lower().endswith('.tif')][0])
    path = f'{DEST}/{fname}'
    open(path, 'wb').write(data)
    with rasterio.open(path, 'r+') as ds:
        a = ds.read(1)
        outside = geometry_mask(AOI_UTM, out_shape=a.shape, transform=ds.transform)   # True = outside the district
        a[outside] = nd
        ds.write(a, 1)
        ds.nodata = nd
        shape = ds.shape
    print(f'  ✔ {fname:24s} {len(data) / 1e6:5.2f} MB  grid {shape}  scale {scale} m')
print('✅ Rasters done. Hansen version used:', HANSEN)

In [ ]:
#@title 4️⃣ Export time series tables (climate, population, built-up, night lights, NDVI)
NY = YEAR_TO - YEAR_FROM + 1
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(f'{YEAR_FROM}-01-01', f'{YEAR_TO + 1}-01-01').select('precipitation')
era = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate(f'{YEAR_FROM}-01-01', f'{YEAR_TO + 1}-01-01').select('temperature_2m')

def mon(m):
    m = ee.Number(m)
    p = chirps.filter(ee.Filter.calendarRange(m, m, 'month')).sum().divide(NY)
    t = era.filter(ee.Filter.calendarRange(m, m, 'month')).mean().subtract(273.15)
    return ee.Feature(None, {'month': m,
                             'rain': p.reduceRegion(ee.Reducer.mean(), AOI, 5566, maxPixels=1e10, bestEffort=True).get('precipitation'),
                             'temp': t.reduceRegion(ee.Reducer.mean(), AOI, 11132, maxPixels=1e10, bestEffort=True).get('temperature_2m')})

def yr(y):
    y = ee.Number(y)
    p = chirps.filter(ee.Filter.calendarRange(y, y, 'year')).sum()
    return ee.Feature(None, {'year': y, 'rain': p.reduceRegion(ee.Reducer.mean(), AOI, 5566, maxPixels=1e10, bestEffort=True).get('precipitation')})

def tab(fc):
    return pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])

mdf = tab(ee.FeatureCollection(ee.List.sequence(1, 12).map(mon))).sort_values('month')
mdf.to_csv(f'{DEST}/climate_monthly.csv', index=False)
tab(ee.FeatureCollection(ee.List.sequence(YEAR_FROM, YEAR_TO).map(yr))).sort_values('year').to_csv(f'{DEST}/rain_annual.csv', index=False)

def yr_sum(y):
    y = ee.Number(y)
    img = pop_c.filter(ee.Filter.eq('year', y)).select('population').mosaic()
    return ee.Feature(None, {'year': y, 'pop': img.reduceRegion(ee.Reducer.sum(), AOI, 100, maxPixels=1e11, bestEffort=True).get('population')})
tab(ee.FeatureCollection(ee.List.sequence(2000, 2020).map(yr_sum))).sort_values('year').to_csv(f'{DEST}/pop_timeseries.csv', index=False)

def bsum(img):
    v = img.reduceRegion(ee.Reducer.sum(), AOI, 100, maxPixels=1e11, bestEffort=True).get('built_surface')
    return ee.Feature(None, {'year': ee.Date(img.get('system:time_start')).get('year'), 'built_m2': v})
tab(gh.map(bsum)).sort_values('year').to_csv(f'{DEST}/built_timeseries.csv', index=False)

def ysum(img):
    yr_ = ee.Number.parse(ee.String(img.get('system:index')).slice(0, 4))
    v = img.max(0).reduceRegion(ee.Reducer.mean(), AOI, 500, maxPixels=1e10, bestEffort=True).get('average')
    return ee.Feature(None, {'year': yr_, 'rad': v})
tab(ntl.map(ysum)).sort_values('year').to_csv(f'{DEST}/ntl_timeseries.csv', index=False)

nd = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
def ny(y):
    y = ee.Number(y)
    img = nd.filter(ee.Filter.calendarRange(y, y, 'year')).mean().multiply(0.0001)
    return ee.Feature(None, {'year': y, 'ndvi': img.reduceRegion(ee.Reducer.mean(), AOI, 1000, maxPixels=1e10, bestEffort=True).get('NDVI')})
tab(ee.FeatureCollection(ee.List.sequence(2001, 2024).map(ny))).dropna().sort_values('year').to_csv(f'{DEST}/ndvi_timeseries.csv', index=False)
print('✅ Tables saved:', sorted(f for f in os.listdir(DEST) if f.endswith('.csv')))

In [ ]:
#@title 5️⃣ Export OpenStreetMap economic features (points)
import osmnx as ox
ox.settings.timeout = 180
tags = {'tourism': ['hotel', 'guest_house', 'attraction', 'viewpoint', 'museum', 'camp_site', 'resort'],
        'landuse': ['quarry', 'industrial', 'commercial', 'retail'],
        'amenity': ['marketplace', 'bank', 'hospital', 'clinic', 'school', 'university'],
        'man_made': ['works'], 'power': ['plant']}
MAPPING = {'quarry': 'Mining / quarry', 'industrial': 'Industry', 'works': 'Industry', 'commercial': 'Trade / commerce', 'retail': 'Trade / commerce',
           'marketplace': 'Trade / commerce', 'bank': 'Finance', 'hospital': 'Health', 'clinic': 'Health', 'school': 'Education',
           'university': 'Education', 'plant': 'Energy'}
def classify(r):
    for k in ['tourism', 'landuse', 'amenity', 'man_made', 'power']:
        if k in r and isinstance(r[k], str):
            return MAPPING.get(r[k], 'Tourism' if k == 'tourism' else None)
    return None
w_, s_, e_, n_ = aoi_union.bounds
try:
    try:
        f = ox.features_from_bbox(bbox=(w_, s_, e_, n_), tags=tags)
    except TypeError:
        f = ox.features_from_bbox(n_, s_, e_, w_, tags=tags)
    f = gpd.GeoDataFrame(f, geometry='geometry', crs=4326)
    f['category'] = f.apply(classify, axis=1)
    f = f[f['category'].notna()].copy()
    f['geometry'] = f.geometry.representative_point()
    f = f[f.geometry.within(aoi_union)]
    f['name'] = f['name'] if 'name' in f.columns else ''
    f[['category', 'name', 'geometry']].to_file(f'{DEST}/osm_features.geojson', driver='GeoJSON')
    print('✅ OSM features:', len(f))
except Exception as e:
    print('⚠️ OSM download failed:', str(e)[:120], '– the student notebook will skip this part.')

In [ ]:
#@title 6️⃣ Write metadata, zip and download
meta = {'study_name': DISTRICT, 'folder': AREA_FOLDER, 'crs': CRS, 'year_from': YEAR_FROM, 'year_to': YEAR_TO,
        'hansen_dataset': HANSEN, 'exported': pd.Timestamp.now().strftime('%Y-%m-%d')}
json.dump(meta, open(f'{DEST}/meta.json', 'w'), indent=1)
tot = sum(os.path.getsize(os.path.join(DEST, f)) for f in os.listdir(DEST)) / 1e6
print(f'Total size: {tot:.1f} MB (limit per GitHub file = 100 MB; this is the whole folder)')
zp = shutil.make_archive(f'/content/{AREA_FOLDER}_data', 'zip', '/content', AREA_FOLDER)
from google.colab import files
files.download(zp)
print('➡️ Unzip, then copy the folder to your repo as data/' + AREA_FOLDER + '/ and push.')